# Plot Example Frames

Extracts example aerial image frames centred on predicted bubble locations, overlays the prediction mask in yellow, and saves each frame as a PNG.

Set paths in the **Configuration** cell before running.

In [ ]:
%matplotlib inline
import rasterio
import rasterio.windows
import rasterio.features
import matplotlib.pyplot as plt
import os
from shapely.geometry import box
import geopandas as gpd
import numpy as np
from PIL import Image

## Configuration

Edit the paths below to match your data.

In [ ]:
# Directory containing .jp2 aerial imagery files
image_dir = '/isipd/projects/p_planetdw/data/dw_detection/examplevis'

# Vector file of predicted bubble polygons
prediction_path = '/isipd/projects/p_planetdw/data/dw_detection/examplevis/dw_larger100m2_connected.gpkg'

# Vector file of centroid points (one per bubble, used to centre each frame)
center_path = '/isipd/projects/p_planetdw/data/dw_detection/examplevis/centroids.gpkg'

# Output directory for saved frame PNGs
output_dir = '/isipd/projects/p_planetdw/data/dw_detection/examplevis/example_frames'

## Helper Functions

In [ ]:
def percentile_stretch(img, lower=2, upper=98):
    """Stretch each band to 0-255 using percentile clipping for display."""
    stretched = np.zeros_like(img, dtype=np.uint8)
    for i in range(img.shape[0]):
        band = img[i]
        p2 = np.percentile(band, lower)
        p98 = np.percentile(band, upper)
        if p98 > p2:
            stretched[i] = np.clip(255 * (band - p2) / (p98 - p2), 0, 255).astype(np.uint8)
        else:
            stretched[i] = np.zeros_like(band, dtype=np.uint8)
    return stretched


def plot_example_frames(image_dir, prediction_path, center_path, output_dir,
                        num_frames=20, extent=1000):
    """Extract frames centred on bubble centroids, overlay predictions, and save as PNGs."""
    centers = gpd.read_file(center_path)
    predictions = gpd.read_file(prediction_path)

    if centers.empty:
        print("No centers found.")
        return
    if predictions.empty:
        print("No predictions found.")
        return

    frames = []
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.jp2')]

    if not image_files:
        print("No .jp2 images found.")
        return

    for idx, center in enumerate(centers.geometry):
        print(f"\nProcessing center {idx + 1}/{len(centers)}")

        for image_name in image_files:
            image_path = os.path.join(image_dir, image_name)
            with rasterio.open(image_path) as src:
                if centers.crs != src.crs:
                    center_transformed = gpd.GeoSeries([center], crs=centers.crs).to_crs(src.crs).iloc[0]
                else:
                    center_transformed = center

                cx, cy = center_transformed.x, center_transformed.y
                bbox = box(cx - extent, cy - extent, cx + extent, cy + extent)
                img_bounds = box(*src.bounds)
                if not bbox.intersects(img_bounds):
                    print("No intersection with image.")
                    continue

                print(f"Reading image: {image_name}")

                row_start, col_start = src.index(bbox.bounds[0], bbox.bounds[3])
                row_stop, col_stop = src.index(bbox.bounds[2], bbox.bounds[1])

                row_start, row_stop = max(0, row_start), min(src.height, row_stop)
                col_start, col_stop = max(0, col_start), min(src.width, col_stop)

                window = rasterio.windows.Window.from_slices(
                    (row_start, row_stop), (col_start, col_stop)
                )
                img = src.read(window=window)

                pred = predictions.to_crs(src.crs)
                pred_crop = pred[pred.geometry.intersects(bbox)]
                if pred_crop.empty:
                    print("No prediction overlap.")
                    continue

                height, width = img.shape[1:]
                pred_raster = np.zeros((height, width), dtype=np.uint8)
                transform = src.window_transform(window)

                for geom in pred_crop.geometry:
                    if geom.is_empty:
                        continue
                    if geom.geom_type == 'Polygon':
                        geom_iter = [geom]
                    elif geom.geom_type == 'MultiPolygon':
                        geom_iter = list(geom.geoms)
                    else:
                        continue

                    for g in geom_iter:
                        try:
                            mask = rasterio.features.rasterize(
                                [(g, 1)],
                                out_shape=(height, width),
                                transform=transform,
                                fill=0,
                                dtype=np.uint8
                            )
                            pred_raster = np.maximum(pred_raster, mask)
                        except Exception as e:
                            print(f"Skipping geometry: {e}")

                if pred_raster.sum() == 0:
                    print("Empty prediction mask after rasterization.")
                    continue

                img = np.concatenate((img, pred_raster[np.newaxis, :, :]), axis=0)
                frames.append(img)

                print("Frame added.")
                if len(frames) >= num_frames:
                    break
        if len(frames) >= num_frames:
            break

    if not frames:
        print("No frames were generated.")
        return

    os.makedirs(output_dir, exist_ok=True)
    fig_handles = []
    for i, frame in enumerate(frames):
        print(f"Saving frame {i} with shape {frame.shape}")

        rgb_stretched = percentile_stretch(frame[[2, 1, 0]])  # PlanetScope: R=2, G=1, B=0
        rgb = np.moveaxis(rgb_stretched, 0, -1)

        pred_mask = frame[-1]
        yellow_overlay = np.zeros((*pred_mask.shape, 4), dtype=np.uint8)
        yellow_overlay[..., 0] = 255
        yellow_overlay[..., 1] = 51
        yellow_overlay[..., 2] = 255
        yellow_overlay[..., 3] = np.where(pred_mask > 0, 255, 0).astype(np.uint8)

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(rgb)
        ax.imshow(yellow_overlay, interpolation='none')
        ax.axis('off')

        out_path = os.path.join(output_dir, f'frame_{i}.png')
        fig.savefig(out_path, bbox_inches='tight', pad_inches=0)
        print(f"Saved to {out_path}")
        fig_handles.append(fig)

    plt.show()
    print(f"\n{len(frames)} frame(s) saved to {output_dir}")

## Run

In [ ]:
os.makedirs(output_dir, exist_ok=True)
plot_example_frames(image_dir, prediction_path, center_path, output_dir)